[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/surface_curvature/surface_curvature.ipynb)

# Curvature of Quadric Surfaces

This example computes the principal curvatures and lines of curvature across smooth quadric surfaces. At every surface point, two quadratic forms on tangent directions measure how the surface bends: the second fundamental form records how rapidly the surface curves away from its tangent plane, and the first fundamental form measures Euclidean tangent lengths, so solving the second against the first yields the extreme curvatures. For any central quadric, the curves that follow those principal directions across the surface coincide with the intersections of the quadric with its confocal family.

A quadric surface is modelled as a polarity, a symmetric map from points to planes sending each point to its polar plane, and the surface is where a point lies on its own polar plane, `surface(point) & point == 0`. Evaluated on zero-weight directions, the bilinear form `surface(Point) & Point` acts directly as the surface's Hessian; restricting both arguments with the tangent projector gives the second fundamental form up to gradient length, whose generalized eigenvalues against the direction metric `Direction.dual() | Direction.dual()` are the principal curvatures. Subtracting a point's offset dyad from the dual quadric `surface.inverse()` restricted to central planes leaves one form whose eigenvalues identify the confocal quadrics passing through that point.

In classical differential-geometry and homogeneous-coordinate notation, a quadric matrix $Q \in \mathbb{R}^{4 \times 4}$ requires choosing local surface coordinates $(u, v) \mapsto \mathbf{r}(u, v)$ to assemble the $2 \times 2$ fundamental matrices $\mathrm{I}_{ij} = \partial_i \mathbf{r} \cdot \partial_j \mathbf{r}$ and $\mathrm{II}_{ij} = \hat{\mathbf{n}} \cdot \partial_i \partial_j \mathbf{r}$ before solving $\det(\mathrm{II} - \kappa\,\mathrm{I}) = 0$, while Dupin's confocal lines of curvature solve the separate Cartesian secular equation $\sum_{k=1}^3 x_k^2 / (a_k^2 - \lambda) = 1$ in the principal-axis frame after shifting to the centre and diagonalizing the $3 \times 3$ quadratic block.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack, where
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure
from examples.surfaces.surface_curvature import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                 # Scalar+
# Finite points carry homogeneous weight, and directions are ideal points of weight zero.
Point = ga.gatype.antivector()                              # Antivector-
Plane = ga.gatype.vector()                                  # Vector+
Direction = Point.derive.degenerate                         # Antivector-
# A polarity maps each point to its polar plane.
Quadric = ga.gatype((Plane, Point))                         # Plane+ <- Point-

# The three coordinate planes x = 0, y = 0, z = 0, and the plane at infinity.
axes = mv(Plane.derive.nondegenerate, [[1, 0, 0], [0, 1, 0], [0, 0, 1]])  # [3] Plane+
w = mv.w                                                    # [] Plane+

# A direction's dual is the plane through the origin normal to it; their inner product is the Euclidean metric.
metric = Direction.dual() | Direction.dual()                # [] Scalar+ <- (Direction-, Direction-)
print(Direction, Quadric, sep="\n")

## Quadrics as polarities from points to planes

Pairing `axes & Point` reads a point's Euclidean coordinates, and multiplying back by `axes` sums the coordinate plane dyads into a polarity that sends every point to its polar plane. A point lies on the quadric surface where it lies on its own polar plane, `surface(point) & point == 0`. Along an orthographic ray, that bilinear form becomes a scalar quadratic in distance whose smaller positive root locates the visible surface point.

In homogeneous matrix notation, the polarity is represented by a symmetric matrix $Q \in \mathbb{R}^{4 \times 4}$ mapping column vectors $\mathbf{x} = (x, y, z, 1)^\top$ to row planes $(Q\mathbf{x})^\top$ with surface locus $\mathbf{x}^\top Q\,\mathbf{x} = 0$.

In [ ]:
# The ellipsoid's semi-axes, longest to shortest, and the hyperboloid's, with one negative sign for a saddle.
semi_axes = np.array([3.0, 2.0, 1.2])
hyperboloid_axes = np.array([1.6, 1.0, 1.0])
hyperboloid_signs = np.array([1.0, 1.0, -1.0])
# Each surface's camera elevation and azimuth in degrees, half the screen width, and the vertical clip height.
views = (((25.0, -60.0), 3.4), ((18.0, -60.0), 3.6))
n_pixels = 360
heights = (np.inf, 2.0)

In [ ]:
def central_quadric(weights: np.ndarray) -> Quadric:
    """Coordinate plane dyads weighted by `weights` (`[3]`) minus the plane-at-infinity dyad form a central quadric (`[] Plane+ <- Point-`)."""
    return (axes * (axes & Point) * weights).sum() - w * (w & Point)                    # [] Plane+ <- Point-


# A negative weight on one axis turns the ellipsoid into a hyperboloid of one sheet.
ellipsoid = central_quadric(1 / semi_axes**2)                                           # [] Plane+ <- Point-
hyperboloid = central_quadric(hyperboloid_signs / hyperboloid_axes**2)                  # [] Plane+ <- Point-

In [ ]:
def nearest_root(quadratic: Scalar, linear: Scalar, discriminant: Scalar) -> Scalar:
    """The smaller positive root of `quadratic * t**2 + 2 * linear * t + constant` gives the first ray step (`[...] Scalar+`), or zero."""
    zero = discriminant * 0                                                             # [...] Scalar+
    root = where(discriminant > 0, discriminant, zero).square_root()                    # [...] Scalar+
    with np.errstate(divide="ignore", invalid="ignore"):
        first, second = (-linear - root) / quadratic, (-linear + root) / quadratic      # [...] Scalar+, Scalar+
    positive = where(second > 0, second, zero)                                          # [...] Scalar+
    smaller = where((first > 0) & (~(second > 0) | (first < second)), first, positive)  # [...] Scalar+
    return where(smaller.isfinite(), smaller, zero)                                     # [...] Scalar+


def ray_intersections(surface: Quadric, origins: Point, heading: Direction) -> tuple[Point, Scalar]:
    """Each ray from `origins` along `heading` meets `surface` at its nearer positive root (`[...] Point-`) with quadratic discriminant (`[...] Scalar+`)."""
    form = surface(Point) & Point                                                       # [] Scalar+ <- (Point-, Point-)
    # Evaluating the bilinear form on a ray gives the coefficients of a quadratic in the distance along it.
    quadratic = form(heading, heading)                                                  # [...] Scalar+
    linear = form(heading, origins)                                                     # [...] Scalar+
    constant = form(origins, origins)                                                   # [...] Scalar+
    discriminant = linear * linear - quadratic * constant                               # [...] Scalar+
    hits = (origins + heading * nearest_root(quadratic, linear, discriminant)).normalized()  # [...] Point-
    return hits, discriminant


rays = [render.view_rays(*view, extent, n_pixels) for view, extent in views]
hits = [ray_intersections(surface, *ray) for surface, ray in zip((ellipsoid, hyperboloid), rays)]
scenes = [
    (points, discriminant, surface(points), heading)
    for surface, (points, discriminant), (_, heading) in zip((ellipsoid, hyperboloid), hits, rays)
]
save_figure(render.surfaces_figure(scenes, heights), "surface_curvature_surfaces");

The left panel shows the triaxial ellipsoid and the right panel shows the hyperboloid of one sheet clipped horizontally at constant height.

## Principal curvatures from the tangent-projected Hessian

Evaluated on ideal directions, `surface(Point) & Point` measures how the quadric's algebraic value changes to second order along a tangent step, independently of a point's homogeneous weight. At any surface point, its polar plane `surface(points)` is the tangent plane, and projecting directions onto that plane along the normal removes the off-surface component. Dividing the projected Hessian by the gradient length produces the second fundamental form on `Direction`, whose eigenvalues against `metric` give the two principal curvatures together with a zero normal eigenvalue.

In classical surface differential geometry, the principal curvatures solve $\det(\mathrm{II} - \kappa\,\mathrm{I}) = 0$ as the eigenvalues of the $2 \times 2$ Weingarten shape operator $\mathrm{I}^{-1}\mathrm{II}$, with Gaussian curvature $K = \det\mathrm{II} / \det\mathrm{I}$.

In [ ]:
# The Gaussian curvature magnitude at which the colour map saturates.
curvature_scale = 1.2

In [ ]:
def tangent_pair(values: Scalar) -> Scalar:
    """Dropping the normal eigenvalue nearest zero from `values` (`[..., 3] Scalar+`) leaves the two ordered tangent eigenvalues (`[..., 2] Scalar+`)."""
    zero_index = values.abs().argmin(axis=-1)
    lower = where(zero_index == 0, values[..., 1], values[..., 0])                      # [...] Scalar+
    upper = where(zero_index == 2, values[..., 1], values[..., 2])                      # [...] Scalar+
    return stack([lower, upper], axis=-1)                                               # [..., 2] Scalar+


def principal_curvatures(surface: Quadric, points: Point) -> Scalar:
    """Ordered principal curvatures (`[..., 2] Scalar+`) come from the tangent-restricted Hessian against `metric`."""
    tangent = surface(points)                                                           # [...] Plane+
    # The normal direction drops the tangent plane's homogeneous weight, and the projector removes its component.
    normal = tangent.dual().cast(Direction)                                             # [...] Direction-
    projector = Direction - normal * ((tangent & Direction) / (tangent & normal))       # [...] Direction- <- Direction-
    # Restricting the Hessian to the tangent plane on both sides and dividing by gradient length gives the second form.
    second_form = (surface(Point) & Point)(projector, projector) / metric(normal, normal).square_root()  # [...] Scalar+ <- (Direction-, Direction-)
    return tangent_pair(second_form.eigvalsh(metric))                                   # [..., 2] Scalar+


curvatures = [principal_curvatures(surface, points) for surface, (points, _) in zip((ellipsoid, hyperboloid), hits)]  # [n_pixels, n_pixels, 2] Scalar+
save_figure(render.gaussian_curvature_figure(scenes, curvatures, heights, curvature_scale), "surface_curvature_gaussian");

Red shades mark positive Gaussian curvature on the ellipsoid where both principal directions bend toward the interior, with the deepest red concentrated at the sharp tips of the longest axis, while blue shades mark negative Gaussian curvature on the saddle-shaped hyperboloid.

In [ ]:
tip_curvatures = principal_curvatures(ellipsoid, mv.point([semi_axes[0], 0.0, 0.0]))    # [2] Scalar+
print("curvatures at the tip of the long axis:", tip_curvatures.to_array(), "  long over each other semi-axis squared:", semi_axes[0] / semi_axes[1:]**2)

## Lines of curvature from the confocal family

Every central quadric belongs to a family of confocal quadrics that share its focal curves and intersect it orthogonally along its lines of curvature. The inverse map `surface.inverse()` sends central planes to poles whose duals define the quadric's intrinsic shape on directions, and confocal members shift that shape by multiples of `metric`. Subtracting a point's own offset dyad leaves a symmetric form on directions that becomes singular whenever a confocal member passes through the point: one eigenvalue vanishes because the point already lies on `surface`, and the other two eigenvalues parameterize the orthogonal net of curvature lines.

In Cartesian principal-axis coordinates, the confocal family of an ellipsoid satisfies $\frac{x^2}{a^2 - \lambda} + \frac{y^2}{b^2 - \lambda} + \frac{z^2}{c^2 - \lambda} = 1$, whose two non-zero roots $\lambda$ at each surface point serve as ellipsoidal coordinates.

In [ ]:
def confocal_parameters(surface: Quadric, points: Point) -> Scalar:
    """Ordered confocal family parameters (`[..., 2] Scalar+`) come from the dual shape minus each point's offset dyad."""
    dual_surface = surface.inverse()                                                    # [] Point- <- Plane+
    # The pole of the plane at infinity is the quadric's centre, normalized to unit weight.
    pole = dual_surface(w)                                                              # [] Point-
    centre = pole / (w & pole)                                                          # [] Point-
    # Shift each direction's dual plane to pass through the centre, and dualize its pole under the dual quadric.
    central_planes = Direction.dual() - w * (Direction.dual() & centre)                 # [] Plane+ <- Direction-
    shape = dual_surface(central_planes).dual()                                         # [] Plane+ <- Direction-
    # Each point's displacement from the centre dualizes to the plane through the origin normal to that offset.
    offset_planes = (points - centre).dual()                                            # [...] Plane+
    # Subtract the point's offset dyad from the shape so the eigenvalues give the confocal quadrics through the point.
    confocal_form = -(shape - offset_planes * (offset_planes & Direction)) & Direction  # [...] Scalar+ <- (Direction-, Direction-)
    return tangent_pair(confocal_form.eigvalsh(metric))                                 # [..., 2] Scalar+


parameters = [confocal_parameters(surface, points) for surface, (points, _) in zip((ellipsoid, hyperboloid), hits)]  # [n_pixels, n_pixels, 2] Scalar+
save_figure(render.curvature_lines_figure(scenes, curvatures, parameters, heights, curvature_scale), "surface_curvature_lines");

Charcoal and dark-red contours trace evenly spaced level sets of the two confocal parameters across both surfaces. On the ellipsoid, the two orthogonal families turn around four umbilic points in the plane of the longest and shortest semi-axes, where the surface curves equally in every tangent direction and the two confocal eigenvalues coincide.

In [ ]:
# The four umbilics lie in the symmetry plane spanned by the longest and shortest semi-axes.
long_axis, middle_axis, short_axis = semi_axes
x_coord = long_axis * np.sqrt((long_axis**2 - middle_axis**2) / (long_axis**2 - short_axis**2))
z_coord = short_axis * np.sqrt((middle_axis**2 - short_axis**2) / (long_axis**2 - short_axis**2))
umbilics = mv.point([
    [x_coord, 0.0, z_coord],
    [-x_coord, 0.0, z_coord],
    [x_coord, 0.0, -z_coord],
    [-x_coord, 0.0, -z_coord],
])                                                                                      # [4] Point-
umbilic_curvatures = principal_curvatures(ellipsoid, umbilics)                          # [4, 2] Scalar+
umbilic_parameters = confocal_parameters(ellipsoid, umbilics)                           # [4, 2] Scalar+
print("principal curvatures at the umbilics:\n", umbilic_curvatures.to_array())
print("confocal parameters at the umbilics:\n", umbilic_parameters.to_array())

In [ ]:
# checks
expected_tip = semi_axes[0] / semi_axes[1:]**2
assert ((tip_curvatures - expected_tip).abs() < 1e-8).all()
# The two principal curvatures and the two confocal parameters agree at the four umbilics.
assert ((umbilic_curvatures[:, 0] - umbilic_curvatures[:, 1]).abs() < 1e-8).all()
assert ((umbilic_parameters[:, 0] - umbilic_parameters[:, 1]).abs() < 1e-6).all()